### **Semana 12: optimización y presupuesto de contexto**

**Objetivo:** comparar calidad y costo sin atribuir capacidades LLM a un recuperador léxico. 

Lectura previa: `Lectura12-CC-0F4.md`.

#### **1. Entorno y evidencia**

El benchmark se ejecuta una vez desde la terminal. Las celdas cargan los resultados reales y verifican hashes.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
base = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'Semana12'/'retrieval_real.py').is_file()), None)
if base is None:
    raise FileNotFoundError('Abrir el cuaderno dentro de CC-0F4')
if str(base) not in sys.path:
    sys.path.insert(0, str(base))
from Semana12 import retrieval_real as real
print('Raíz del curso:', base)
if not (real.RESULTS/'resumen.json').is_file():
    raise FileNotFoundError('Ejecutar una vez en Docker: python -m Semana12.retrieval_real --run')
resumen = real.load_results()
test = pd.read_csv(real.RESULTS/'resultados_todas_las_consultas_test.csv')
calibracion = pd.read_csv(real.RESULTS/'calibracion_solo_desarrollo.csv')
assert len(test) == resumen['split']['test'] and test.question_id.is_unique
print('Resultados verificados, no se repite el ranking.')

In [ ]:
docs, questions, provenance = real.load_squad()
print(provenance['sha256'], provenance['n_articles'], len(docs), len(questions))
display(questions[['question_id','article','question']].head(3))

#### **2. Políticas y función objetivo**

A0 y A1 son top-1 y top-5. F2, F3 y F4 completan la familia fija. B usa el margen entre las dos primeras similitudes para elegir top-1 o top-5.

$$U_\lambda=\overline H-\lambda\overline k.$$

El acierto se refiere al párrafo de origen, no a una respuesta LLM.

In [ ]:
display(pd.DataFrame(resumen['test_results']).T)
assert (test.hit_top1 <= test.hit_B_adaptive).all()
assert (test.hit_B_adaptive <= test.hit_top5).all()
print('Baseline fijo seleccionado solo en desarrollo:', resumen['best_fixed_on_development'])

#### **3. Calibración y estabilidad**

Separamos artículos, no preguntas. El índice ve todos los textos sin etiquetas. El test no selecciona umbral. Examina el margen de utilidad entre candidatos.

In [ ]:
display(calibracion)
print('Diferencia de utilidad entre los dos mejores:', resumen['dev_gap_top2'])
semillas = pd.read_csv(real.RESULTS/'sensibilidad_semillas.csv')
display(semillas[semillas.policy == real.ADAPTIVE])

#### **4. Frontera de costo y comparación justa**

Cada política congelada define una recta. La envolvente superior depende de lambda. La B recalibrada usa desarrollo para cada lambda.No se escoge mirando el test.

In [ ]:
curvas = pd.read_csv(real.RESULTS/'sensibilidad_lambda.csv')
ax = curvas.pivot(index='lambda',columns='policy',values='utility').plot(figsize=(9,4))
ax.set(xlabel='Lambda',ylabel='Utilidad media de prueba')
plt.show()
display(pd.DataFrame(resumen['lambda_frontier_frozen_test']))
display(pd.Series(resumen['equal_budget_mix_test']))

#### **5. Incertidumbre y diferencias**

El micro pondera preguntas y el macro artículos. El bootstrap emparejado por artículo recalcula el denominador. Las bandas son exploratorias.

In [ ]:
display(pd.DataFrame(resumen['test_article_cluster_CI95']).T)

#### **6. Entrenamiento frente a inferencia**

SFT actualiza pesos. LoRA usa $W'=W+BA$ y QLoRA cuantiza los pesos base. Test-time compute incrementa recursos durante la inferencia. Ninguno de estos mecanismos es evaluado por el benchmark de contexto.

In [ ]:
d, k, r = 4096, 4096, 8
print('Parámetros de matriz completa:', d*k)
print('Parámetros de adaptador:', r*(d+k))

#### **Cierre**

Explica por qué recuperar el párrafo de origen no demuestra una respuesta correcta. Las preguntas se redactaron viendo el párrafo y el test es interno al dev público. Los análisis v2 son secundarios exploratorios.